# BioJEPA v0.7 Alignment HPO v10

Short-horizon search: 4000 train epochs with OneCycleLR schedule sized for 10000 epochs. Trains through full warmup + first ~37% of cooldown, evaluating configs at the same LR state. 20 trials with widened ranges, seeded with v9 winner and v0.6 values as anchors.

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import types
import gc
import sys
from pathlib import Path
from collections import Counter
from functools import partial
import numpy as np
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from hpo_utils_alignment import (
    is_valid_alignment_config,
    compute_alignment_objective, summarize_alignment_results,
    run_selected_alignment_evals,
)

sys.path.insert(0, str(Path.cwd().parent))

from biojepa_v0_7 import ActionComposer, ActionComposerConfig
from training_v0_7 import load_feature_banks, get_seq_embeddings, get_target_embeddings, reset_seed
from dataloader_v0_7 import ComposerLoader
from config_v0_7 import DataConfig
from evals.evals import EvalContext


## Device & Paths

In [ ]:
def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(1337)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(1337)
random.seed(1337)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_FUSED = torch.cuda.is_available()

data_root = Path('~/data/v0_7').expanduser()
ref_root = Path('~/data/reference_data').expanduser()
hpo_output_dir = data_root / 'hpo'
hpo_output_dir.mkdir(exist_ok=True)


## Load Feature Banks

In [ ]:
data_cfg = DataConfig(data_root=data_root, checkpoint_dir=data_root / 'checkpoints', ref_dir=ref_root)
seq_banks, target_bank = load_feature_banks(data_cfg, device)


## HPO Config

In [ ]:
version = 'v10'


In [ ]:
hpo_config = {
    'study_name': f'biojepa_v0_7_alignment_hpo_{version}',
    'seed': 1337,
    'verbose': False,
    'n_startup_trials': 5,
    'multivariate': True,
    'constant_liar': True,
    'pruner_startup_trials': 5,
    'pruner_warmup_steps': 1,

    # Fixed encoder params (T14 winner, required by EvalContext)
    'num_genes': 10000,
    'embed_dim': 256,
    'n_layer': 6,
    'heads': 4,

    # Short-horizon search: train for 4000 epochs but use a 10000-epoch OneCycleLR.
    # At step 4000*184=736k, LR is at ~80% of max (37% through cooldown).
    # Configs are compared at matched schedule state, just truncated.
    'epochs': 4000,
    'lr_schedule_epochs': 10000,
    'cheap_eval_epochs': 500,
    'full_eval_epochs': 1000,
    'warmup_epochs': 1000,

    # Fixed from prior studies
    'pert_latent_dim': 128,
    'pert_mode_dim': 64,
    'batch_size': 64,
    'chemical_fraction': 0.1,

    # v10: widened ranges on all three axes vs v9.
    # Brackets v9 winner (lr=3.6e-4, wd=0.003, temp=0.00126) AND v0.6 (lr=7.6e-4, wd=0.011, temp=0.012)
    # with more margin on both ends since the v9 winner did not transfer to production.
    'lr_range': (1e-4, 2e-3),
    'weight_decay_range': (5e-4, 0.03),
    'temperature_range': (3e-4, 0.03),

    'use_amp': USE_AMP,
    'use_fused': USE_FUSED,

    'cheap_evals': ['paired_alignment_quality', 'seq_to_target_retrieval'],
    'full_evals': [
        'seq_to_target_retrieval', 'paired_alignment_quality',
        'cross_modality_target_consistency',
        'mode_sensitivity', 'mode_semantic_consistency',
        'target_family_probing', 'missing_data_robustness',
    ],
}


## Study Setup

In [ ]:
def create_study(cfg):
    study_name = cfg['study_name']
    sampler = TPESampler(
        n_startup_trials=cfg['n_startup_trials'],
        multivariate=cfg['multivariate'],
        constant_liar=cfg['constant_liar'],
        seed=cfg['seed'],
    )
    pruner = MedianPruner(
        n_startup_trials=cfg['pruner_startup_trials'],
        n_warmup_steps=cfg['pruner_warmup_steps'],
        interval_steps=1,
    )
    return optuna.create_study(
        direction='maximize',
        sampler=sampler,
        pruner=pruner,
        study_name=study_name,
        storage=f'sqlite:///{hpo_output_dir / study_name}.db',
        load_if_exists=True,
    )


## Model Factory

In [ ]:
def create_composer(params, cfg, device):
    composer_cfg = ActionComposerConfig(
        latent_dim=params['pert_latent_dim'],
        mode_dim=params['pert_mode_dim'],
        heads=4,
    )
    return ActionComposer(composer_cfg).to(device)


## Objective

In [ ]:
import io, contextlib

def objective(trial, cfg):
    reset_seed(cfg['seed'])

    params = {
        'pert_latent_dim': cfg['pert_latent_dim'],
        'pert_mode_dim': cfg['pert_mode_dim'],
        'batch_size': cfg['batch_size'],
        'lr': trial.suggest_float('lr', *cfg['lr_range'], log=True),
        'weight_decay': trial.suggest_float('weight_decay', *cfg['weight_decay_range']),
        'temperature': trial.suggest_float('temperature', *cfg['temperature_range'], log=True),
    }

    if not is_valid_alignment_config(params):
        trial.set_user_attr('fail_reason', f'Invalid config: pert_latent_dim={params["pert_latent_dim"]} not divisible by 4')
        raise optuna.TrialPruned('Invalid configuration')

    batch_size = params['batch_size']
    temperature = params['temperature']
    chemical_fraction = cfg['chemical_fraction']
    pert_dir = data_root / 'pert_embd'
    composer, train_loader, val_loader, eval_ctx = None, None, None, None
    optimizer, scheduler = None, None

    try:
        composer = create_composer(params, cfg, device)

        with contextlib.redirect_stdout(io.StringIO()):
            train_loader = ComposerLoader(batch_size, 'train', pert_dir, device, seed=cfg['seed'],
                                          chemical_fraction=chemical_fraction)
            val_loader = ComposerLoader(batch_size, 'val', pert_dir, device, seed=cfg['seed'] + 1)

        steps_per_epoch = train_loader.total_samples // batch_size
        train_steps = cfg['epochs'] * steps_per_epoch
        lr_total_steps = cfg['lr_schedule_epochs'] * steps_per_epoch
        print(f'Trial {trial.number}: lr={params["lr"]:.4e} wd={params["weight_decay"]:.3f} '
              f'temp={temperature:.4f} '
              f'| {steps_per_epoch} steps/epoch, {train_steps} train steps ({cfg["epochs"]} epochs), '
              f'LR schedule sized for {lr_total_steps} steps ({cfg["lr_schedule_epochs"]} epochs)')

        eval_config = {
            'num_genes': cfg['num_genes'], 'embed_dim': cfg['embed_dim'],
            'n_layer': cfg['n_layer'], 'heads': cfg['heads'], 'batch_size': 32,
            'pert_latent_dim': params['pert_latent_dim'],
            'pert_mode_dim': params['pert_mode_dim'],
            'verbose': cfg['verbose'], 'seed': cfg['seed'],
        }

        use_autocast = cfg['use_amp'] and device.type == 'cuda'
        fused = cfg['use_fused'] and torch.cuda.is_available()

        optimizer = torch.optim.AdamW(composer.parameters(), lr=params['lr'], weight_decay=params['weight_decay'], fused=fused)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=params['lr'], total_steps=lr_total_steps, pct_start=0.05)

        report_step = 0
        epoch_loss_sum, epoch_loss_count = 0.0, 0
        composer.train()

        for step in range(train_steps):
            b = train_loader.next_batch()
            B = b.seq_idx.shape[0]

            seq_emb = get_seq_embeddings(b.seq_idx.unsqueeze(1), b.modality.unsqueeze(1), seq_banks)
            target_emb = get_target_embeddings(b.target_idx.unsqueeze(1), target_bank)
            mode_ids = b.mode.unsqueeze(1)
            modality_ids = b.modality.unsqueeze(1)
            pert_mask = torch.ones(B, 1, dtype=torch.bool, device=device)

            optimizer.zero_grad()
            with torch.autocast('cuda', dtype=torch.bfloat16, enabled=use_autocast):
                z_seq = composer.encode_sequence_path(seq_emb, modality_ids, mode_ids, pert_mask)
                z_target = composer.encode_target_path(target_emb, mode_ids, pert_mask)
                z_seq = z_seq.squeeze(1)
                z_target = z_target.squeeze(1)
                z_seq = F.normalize(z_seq, dim=1)
                z_target = F.normalize(z_target, dim=1)
                logits = torch.matmul(z_seq, z_target.T) / temperature
                labels = torch.arange(B, device=device)
                loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()

            epoch_loss_sum += loss.item()
            epoch_loss_count += 1

            is_epoch_boundary = (step + 1) % steps_per_epoch == 0
            if not is_epoch_boundary:
                continue

            current_epoch = (step + 1) // steps_per_epoch
            avg_loss = epoch_loss_sum / epoch_loss_count
            epoch_loss_sum, epoch_loss_count = 0.0, 0

            is_cheap_eval = (current_epoch % cfg['cheap_eval_epochs'] == 0)
            is_full_eval = (current_epoch % cfg['full_eval_epochs'] == 0)

            if not (is_cheap_eval or is_full_eval):
                if (current_epoch + 1) % 250 == 0:
                    print(f'  epoch {current_epoch} | loss:{avg_loss:.4f}')
                continue

            evals_to_run = cfg['full_evals'] if is_full_eval else cfg['cheap_evals']

            composer.eval()
            with contextlib.redirect_stdout(io.StringIO()):
                eval_ctx = EvalContext(eval_config, data_root, data_root, ref_root)
                eval_ctx._biojepa = types.SimpleNamespace(composer=composer)
                eval_ctx._seq_banks = seq_banks
                eval_ctx._target_bank = target_bank
                eval_results = run_selected_alignment_evals(eval_ctx, evals_to_run)

            summary = summarize_alignment_results(eval_results)
            trial.set_user_attr(f'eval_epoch_{current_epoch}', summary)
            eval_tag = 'FULL' if is_full_eval else 'cheap'
            print(f'  epoch {current_epoch} | loss:{avg_loss:.4f} | {eval_tag} evals: {summary}')

            if is_full_eval:
                score = compute_alignment_objective(eval_results)
                trial.report(float(score), report_step)
                report_step += 1
                if trial.should_prune():
                    raise optuna.TrialPruned()

            eval_ctx._biojepa = None
            eval_ctx._alignment_inference = None
            eval_ctx = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            composer.train()

        composer.eval()
        with contextlib.redirect_stdout(io.StringIO()):
            eval_ctx = EvalContext(eval_config, data_root, data_root, ref_root)
            eval_ctx._biojepa = types.SimpleNamespace(composer=composer)
            eval_ctx._seq_banks = seq_banks
            eval_ctx._target_bank = target_bank
            final_results = run_selected_alignment_evals(eval_ctx, cfg['full_evals'])

        final_summary = summarize_alignment_results(final_results)
        final_score = compute_alignment_objective(final_results)
        trial.set_user_attr('eval_final', final_summary)
        print(f'  FINAL | score:{final_score:.4f} | {final_summary}')
        return float(final_score)

    except optuna.TrialPruned:
        raise
    except Exception as e:
        import traceback
        trial.set_user_attr('fail_reason', f'{type(e).__name__}: {e}')
        trial.set_user_attr('traceback', traceback.format_exc())
        raise

    finally:
        if eval_ctx is not None:
            eval_ctx._biojepa = None
        eval_ctx = None
        composer, train_loader, val_loader, optimizer, scheduler = None, None, None, None, None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


## Execute

In [ ]:
study = create_study(hpo_config)

# v10: seed v9 winner (HPO-tuned) and v0.6 values (field-proven) as TPE anchors.
v9_winner = {'lr': 3.6e-4, 'weight_decay': 0.003, 'temperature': 0.00126}
v06_values = {'lr': 7.6e-4, 'weight_decay': 0.011, 'temperature': 0.012}
for seed_params in [v9_winner, v06_values]:
    study.enqueue_trial(seed_params, skip_if_exists=True)

study.optimize(
    partial(objective, cfg=hpo_config),
    n_trials=20,
    show_progress_bar=True,
)


## Trial Summary

In [ ]:
for trial in study.trials:
    state = trial.state.name
    reason = trial.user_attrs.get('fail_reason', 'N/A')
    p = trial.params
    print(f'Trial {trial.number}: {state} | '
          f'lr={p.get("lr", 0):.4e} wd={p.get("weight_decay", 0):.3f} '
          f'temp={p.get("temperature", 0):.4f} '
          f'| {reason}')

states = Counter(t.state.name for t in study.trials)
reasons = Counter(t.user_attrs.get('fail_reason', 'N/A') for t in study.trials if t.state.name != 'COMPLETE')
print(f'\nState counts: {dict(states)}')
print(f'Fail reasons: {dict(reasons)}')


## Best Config

In [ ]:
completed_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
if completed_trials:
    print(f'Best trial: {study.best_trial.number}')
    print(f'Best score: {study.best_value:.4f}')
    print(f'Best params: {study.best_params}')
    print(f'\nFinal evals: {study.best_trial.user_attrs.get("eval_final", {})}')
else:
    print('No completed trials.')
